In [1]:
# CELL 1: INSTALL LIBRARIES & IMPORT
!pip install pytesseract -q
!apt-get update -qq && apt-get install -y tesseract-ocr > /dev/null 2>&1

from google.colab import files
from PIL import Image
import cv2
import numpy as np
import pandas as pd
import pytesseract
import re
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import joblib

print('✓ All libraries imported and installed successfully!')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✓ All libraries imported and installed successfully!


In [2]:
# CELL 2: LOAD AND PREPROCESS DATA
print('[CELL 2] Loading and preprocessing data...')

print('\nUpload CSV file with mobile phone data:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print(f'✓ Data loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')

# Use Price(₹) as regression target
TARGET_COL = 'Price(₹)'

if TARGET_COL not in df.columns:
    raise KeyError(
        f"Target column '{TARGET_COL}' not found. "
        f"Available columns: {list(df.columns)}"
    )

# Drop rows with missing price, reset index
df = df.dropna(subset=[TARGET_COL])
df = df.reset_index(drop=True)

print('✓ CELL 2 COMPLETED')


[CELL 2] Loading and preprocessing data...

Upload CSV file with mobile phone data:


Saving ML_READY_MOBILE_DATASET_10000_ROWS (final).csv to ML_READY_MOBILE_DATASET_10000_ROWS (final).csv
✓ Data loaded: (10000, 13)
Columns: ['Brand', 'Series', 'Variant', 'RAM(GB)', 'Storage(GB)', 'Platform', 'Condition', 'Battery(mAh)', 'Camera(MP)', 'Release Year', 'Reviews', 'Ratings', 'Price(₹)']
✓ CELL 2 COMPLETED


In [3]:
# CELL 3: FEATURE ENCODING & PREPARATION
print('[CELL 3] Encoding features...')

if 'df' not in globals():
    raise RuntimeError("df is not defined. Run CELL 2 before CELL 3.")

# Target is the price column from your CSV
TARGET_COL = 'Price(₹)'   # must match df.columns exactly

if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found in df.")

# Separate features and target
X = df.drop(TARGET_COL, axis=1)
y = df[TARGET_COL]

# Encode any categorical (object) columns
encoders = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le
        print(f'Encoded column: {col}')

print(f'X dtypes after encoding:\n{X.dtypes}')
print('✓ CELL 3 COMPLETED')


[CELL 3] Encoding features...
Encoded column: Brand
Encoded column: Series
Encoded column: Variant
Encoded column: Platform
Encoded column: Condition
X dtypes after encoding:
Brand             int64
Series            int64
Variant           int64
RAM(GB)           int64
Storage(GB)       int64
Platform          int64
Condition         int64
Battery(mAh)      int64
Camera(MP)        int64
Release Year      int64
Reviews           int64
Ratings         float64
dtype: object
✓ CELL 3 COMPLETED


In [4]:
print(X.dtypes)


Brand             int64
Series            int64
Variant           int64
RAM(GB)           int64
Storage(GB)       int64
Platform          int64
Condition         int64
Battery(mAh)      int64
Camera(MP)        int64
Release Year      int64
Reviews           int64
Ratings         float64
dtype: object


In [5]:
# CELL 4: TRAIN MODEL
print('[CELL 4] Training model...')

# Safety checks
if 'X' not in globals() or 'y' not in globals():
    raise RuntimeError("X or y not defined. Run CELL 3 before CELL 4.")

# 1) Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2) Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3) XGBoost regressor
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

print('✓ Model trained successfully')
print('✓ CELL 4 COMPLETED')
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_train_pred = xgb_model.predict(X_train_scaled)
train_r2 = r2_score(y_train, y_train_pred)
print(f"Training R² (accuracy): {train_r2:.4f}")

print("✓ Model trained successfully")
print("✓ CELL 4 COMPLETED")


[CELL 4] Training model...
✓ Model trained successfully
✓ CELL 4 COMPLETED
Training R² (accuracy): 0.9999
✓ Model trained successfully
✓ CELL 4 COMPLETED


In [6]:
# CELL 4.1: MODEL EVALUATION (REGRESSION METRICS)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('[CELL 4.1] Evaluating model...')

# Use the same scaled test set from Cell 4
y_pred = xgb_model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2   = r2_score(y_test, y_pred)

print(f"MAE  (Mean Absolute Error): {mae:.4f}")
print(f"MSE  (Mean Squared Error):  {mse:.4f}")
print(f"RMSE (Root MSE):            {rmse:.4f}")
print(f"R²   (R-squared score):     {r2:.4f}")
print('✓ CELL 4.1 COMPLETED')


[CELL 4.1] Evaluating model...
MAE  (Mean Absolute Error): 1482.8962
MSE  (Mean Squared Error):  4436542.5000
RMSE (Root MSE):            2106.3102
R²   (R-squared score):     0.9968
✓ CELL 4.1 COMPLETED


In [7]:
# CELL 5: IMAGE → OCR TEXT
print('[CELL 5] Image processing with OCR...')

if 'files' not in globals():
    raise RuntimeError("Cell 1 not run. Please run CELL 1 (imports) before CELL 5.")

print('\nUpload mobile phone image for prediction:')
# Upload image
uploaded_img = files.upload()
...


[CELL 5] Image processing with OCR...

Upload mobile phone image for prediction:


Saving Screenshot 2025-12-25 152137.png to Screenshot 2025-12-25 152137.png


Ellipsis

In [8]:
# CELL 5: IMAGE → OCR TEXT
print('[CELL 5] Image processing with OCR...')
print('\nUpload mobile phone image for prediction:')

# Upload image
uploaded_img = files.upload()
if not uploaded_img:
    raise ValueError("No image uploaded.")

img_name = list(uploaded_img.keys())[0]
print(f'✓ Image uploaded: {img_name}')

# Open and convert image
pil_img = Image.open(img_name).convert('RGB')
img = np.array(pil_img)
img_cv = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

# Preprocess for better OCR
gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
gray = cv2.medianBlur(gray, 3)
th = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    31, 10
)

# Run Tesseract OCR
custom_config = r'--oem 3 --psm 6'
ocr_text = pytesseract.image_to_string(th, config=custom_config)

print('\n===== OCR TEXT (first 1000 chars) =====')
print(ocr_text[:1000])
print('======================================')

if not ocr_text.strip():
    raise ValueError("OCR produced empty text. Try a clearer image.")

# Make available for Cell 6
globals()['ocr_text'] = ocr_text
print('✓ CELL 5 COMPLETED (ocr_text ready)')


[CELL 5] Image processing with OCR...

Upload mobile phone image for prediction:


Saving Screenshot 2025-12-25 152137.png to Screenshot 2025-12-25 152137 (1).png
✓ Image uploaded: Screenshot 2025-12-25 152137 (1).png

===== OCR TEXT (first 1000 chars) =====
|v | O tpkart Search x j & Apple Phone 16 Bhek, 178.63) x | qa x
( hitps//weew flipkart.com/apple iphone 16 black 128 qb/phtmb0/d6/1998771 2d) MOBHADOY GRNKE RDY#id LST MOBHADOFGENKERDYNBDO/IBotracker_dp banner 1 &bann 7 Qe hat
x F =
aren a
GAIN sic tx products, bands and more a} (on ) commen ap OGD
Electronics + TVs & Appiiances » Mon v Women ¥ Baby & Kids » Home & Furniture + Sports, Books & More + Flights Offer Zone
a Homa > Mobiles & A... > Mobiles > Appia Mobiins > Appla Pron 1) Compare @ Share
C . [ ae } wv Apple iPhone 16 (Black, 128 GB)
aa <0 QED 1.95.286 Ratings & 6,224 Reviews Assured
© oo
CO. %69,900 o
— * £196 Protect Promise Fee Learn more
. i | Secure delivery by 28 Dec, Sunday
. . 4s YQ 7 Available offers
ro]. S 4 .
toy = @ Bank Offer 5% cashback on Axis Bank Flipkart Debit Card up 10. 2/50 TEC
SS 

In [9]:
# CELL 6: EXTRACT FEATURES FROM OCR TEXT
print('[CELL 6] Extracting features from OCR text...')

# Safety checks
if 'X' not in globals():
    raise RuntimeError("X not defined. Run CELL 3 before CELL 6.")
if 'X_train' not in globals():
    raise RuntimeError("X_train not defined. Run CELL 4 before CELL 6.")
if 'ocr_text' not in globals() or not ocr_text:
    raise ValueError("ocr_text missing. Run CELL 5 before CELL 6.")

text_lower = ocr_text.lower()
print(f'Extracted text length: {len(text_lower)} chars')

# Start with None for all model features
features_dict = {col: None for col in X.columns}

def find_first_in_list(text, options):
    for opt in options:
        if opt in text:
            return opt
    return None

# ---------- BRAND ----------
brand_options = [
    'apple', 'google', 'motorola', 'nothing', 'oneplus',
    'oppo', 'poco', 'realme', 'samsung', 'vivo', 'xiaomi'
]

brand_text = find_first_in_list(text_lower, brand_options)
if brand_text:
    # normalize first letter to match LabelEncoder classes ('Apple', 'Samsung', ...)
    brand_text_norm = brand_text.capitalize()
    print(f'Detected brand (text): {brand_text_norm}')

    if 'Brand' in encoders and 'Brand' in X.columns:
        brand_encoder = encoders['Brand']
        if brand_text_norm in brand_encoder.classes_:
            features_dict['Brand'] = int(
                brand_encoder.transform([brand_text_norm])[0]
            )
        else:
            # unseen brand → most frequent brand from training
            features_dict['Brand'] = int(X['Brand'].mode()[0])
    elif 'Brand' in X.columns:
        features_dict['Brand'] = int(X['Brand'].mode()[0])

# ---------- RAM ----------
ram_match = re.search(r'(\d+)\s*gb', text_lower)
if ram_match:
    ram_val = int(ram_match.group(1))
    print(f'Detected RAM: {ram_val} GB')
    if 'RAM(GB)' in X.columns:
        features_dict['RAM(GB)'] = ram_val

# ---------- STORAGE ----------
storage_match = re.search(r'(\d+)\s*(gb|tb)', text_lower)
if storage_match:
    storage_val = int(storage_match.group(1))
    if storage_match.group(2) == 'tb':
        storage_val *= 1024
    print(f'Detected Storage: {storage_val} GB')
    if 'Storage(GB)' in X.columns:
        features_dict['Storage(GB)'] = storage_val

# ---------- RELEASE YEAR (optional) ----------
year_match = re.search(r'20(1[0-9]|2[0-9])', text_lower)
if year_match and 'Release Year' in X.columns:
    year_val = int(year_match.group(0))
    print(f'Detected Release Year: {year_val}')
    features_dict['Release Year'] = year_val

# ---------- FILL REMAINING NUMERIC FEATURES WITH MEDIANS ----------
numeric_medians = X_train.median()
for col in features_dict:
    if features_dict[col] is None and col in numeric_medians.index:
        features_dict[col] = numeric_medians[col]

print('\n✓ features_dict (after extraction + auto-fill):')
for k, v in features_dict.items():
    print(f'  {k}: {v}')

globals()['features_dict'] = features_dict
print('✓ CELL 6 COMPLETED')


[CELL 6] Extracting features from OCR text...
Extracted text length: 1510 chars
Detected brand (text): Apple
Detected RAM: 128 GB
Detected Storage: 128 GB

✓ features_dict (after extraction + auto-fill):
  Brand: 0
  Series: 32.5
  Variant: 4.0
  RAM(GB): 128
  Storage(GB): 128
  Platform: 3.0
  Condition: 1.0
  Battery(mAh): 5200.0
  Camera(MP): 64.0
  Release Year: 2023.0
  Reviews: 5058.5
  Ratings: 4.3
✓ CELL 6 COMPLETED


In [10]:
# See how brands are encoded
brand_le = encoders['Brand']
print(list(brand_le.classes_))      # list of brand names
print(brand_le.transform(brand_le.classes_))  # their numeric codes


['Apple', 'Google', 'Motorola', 'Nothing', 'OnePlus', 'Oppo', 'Poco', 'Realme', 'Samsung', 'Vivo', 'Xiaomi']
[ 0  1  2  3  4  5  6  7  8  9 10]


In [11]:
# CELL 7: PREDICT PRICE FROM OCR FEATURES
print('[CELL 7] Predicting price from extracted features...')

# Safety checks
if 'features_dict' not in globals():
    raise RuntimeError("features_dict missing. Run CELL 6 before CELL 7.")
if 'X' not in globals() or 'scaler' not in globals() or 'xgb_model' not in globals():
    raise RuntimeError("Model objects missing. Run CELL 3 and CELL 4 before CELL 7.")

# 1) Build a single-row DataFrame using same columns as X
pred_df = pd.DataFrame([features_dict], columns=X.columns)

# Ensure all columns exist and are ordered correctly
for col in X.columns:
    if col not in pred_df.columns:
        pred_df[col] = 0

pred_df = pred_df[X.columns]

print('\nInput row for prediction:')
print(pred_df)

# 2) Scale features using training scaler
pred_scaled = scaler.transform(pred_df)

# 3) Predict price
pred_price = xgb_model.predict(pred_scaled)[0]

print(f'\nPredicted Price(₹) for this phone: ₹ {pred_price:,.2f}')
print('✓ CELL 7 COMPLETED')


[CELL 7] Predicting price from extracted features...

Input row for prediction:
   Brand  Series  Variant  RAM(GB)  Storage(GB)  Platform  Condition  \
0      0    32.5      4.0      128          128       3.0        1.0   

   Battery(mAh)  Camera(MP)  Release Year  Reviews  Ratings  
0        5200.0        64.0        2023.0   5058.5      4.3  

Predicted Price(₹) for this phone: ₹ 65,920.25
✓ CELL 7 COMPLETED
